# 01 — Data Ingestion

**Goal:** Pull all raw data sources into Google Drive, filtered to NPM only. Everything runs inside Colab — no manual file uploads needed.

## Data source strategy

| Source | What we use it for | How we access it |
|---|---|---|
| **`deps.dev` (BigQuery)** | NPM dependency edges — current, Google-maintained | BigQuery public dataset, free |
| **Libraries.io (Kaggle API)** | Package metadata: stars, repo URLs, dependent counts | `kaggle` CLI, free with account |
| **GitHub Archive (BigQuery)** | Contributor commit activity for Bus Factor | BigQuery public dataset, free |
| **OSV npm feed** | Known CVEs per package | Direct HTTP download, no auth |
| **npm registry API** | Maintainer publish rights for Orphan Risk | Public REST API |

## Why deps.dev instead of Libraries.io for dependencies?
The Libraries.io CSV snapshot on Kaggle is from 2020 — four years out of date. The npm ecosystem moves fast; popular packages like `vite` and `turbo` didn't even exist then. `deps.dev` (`bigquery-public-data.deps_dev_v1`) is maintained by Google's Open Source Insights team, updated weekly, and is queryable directly from Colab via BigQuery with no file handling. It's what Google's own security team uses for supply chain analysis.

Libraries.io is still valuable for its metadata (star counts, repo URLs) which `deps.dev` doesn't provide, so we use it as a supplement via the Kaggle API.

## What this notebook produces
| Output file | Contents |
|---|---|
| `processed/npm_packages.parquet` | NPM package metadata (merged deps.dev + Libraries.io) |
| `processed/npm_dependencies.parquet` | NPM dependency edges from deps.dev |
| `processed/gh_commits.parquet` | GitHub Archive contributor activity |
| `processed/osv_npm.parquet` | OSV vulnerabilities, parsed to flat table |
| `processed/npm_maintainers.parquet` | npm registry maintainer + publish-date data |
| `sample/npm_packages_10k.parquet` | Top 10k packages by dependent count (for fast dev) |
| `sample/npm_dependencies_10k.parquet` | Dependency edges filtered to the 10k sample |
| `sample/gh_commits_10k.parquet` | Commits filtered to the 10k sample |

## Learning note
Each section has a **Why** block explaining the design decision. Read those before running the cells.

## 0 — Setup: Git, Drive, and project paths

**Why `.env` + fallback to `userdata`?**
The VS Code Colab extension can't access Colab Secrets (`userdata.get`) because that flow requires the Colab browser UI. Instead we load secrets from a `.env` file sitting next to the notebooks in the repo. The `.env` file is in `.gitignore` so it's never committed. If `.env` isn't found (e.g. someone runs directly in the Colab UI), we fall back to `userdata.get` automatically.

**One-time setup:** Copy `.env.example` → `.env` in the repo root and fill in your values.

In [ ]:
!pip install -q pandas pyarrow requests aiohttp tqdm google-cloud-bigquery db-dtypes nest_asyncio kaggle python-dotenv

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the repo root (one level up from notebooks/)
env_path = Path(__file__).parent.parent / '.env' if '__file__' in dir() else Path('/content/drive/MyDrive/projects/BlastRadius/.env')
load_dotenv(dotenv_path=env_path)

def get_secret(key):
    """Load from .env first (VS Code Colab extension), fall back to Colab userdata."""
    value = os.getenv(key)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(key)
    except Exception:
        raise EnvironmentError(f"Secret '{key}' not found in .env or Colab Secrets.")

PROJECT_ROOT = get_secret('BLAST_RADIUS_PATH')

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Wire up sys.path so we can import from utils/
for path in [PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'utils')]:
    if path and path not in sys.path and os.path.exists(path):
        sys.path.append(path)

from utils.config import initialize_project
initialize_project()

# Create data subdirectories if they don't exist yet
for subdir in ['data/raw/osv_npm', 'data/processed', 'data/sample']:
    os.makedirs(f'{PROJECT_ROOT}/{subdir}', exist_ok=True)

## 1 — Authenticate with Google Cloud

**Why:** We use BigQuery for two datasets: `deps.dev` (dependencies) and `githubarchive` (commits). Both are free public datasets. Colab Pro has GCP built in — `authenticate_user()` does a one-time browser OAuth flow. No service account key file needed.

**Secrets needed** (set via the key icon in the Colab left sidebar):
- `BLAST_RADIUS_PATH` — e.g. `/content/drive/MyDrive/projects/BlastRadius`
- `GCP_BLAST_RADIUS-ID` — your GCP project ID, e.g. `blast-radius-494118`

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('GCP authentication complete.')

In [ ]:
from google.cloud import bigquery

GCP_PROJECT_ID = get_secret('GCP_BLAST_RADIUS_ID')
bq = bigquery.Client(project=GCP_PROJECT_ID)
print(f'BigQuery client ready. Project: {GCP_PROJECT_ID}')

## 2 — deps.dev: NPM dependency graph (BigQuery)

**Why deps.dev?** `bigquery-public-data.deps_dev_v1` is Google's Open Source Insights dataset — the same one that powers [deps.dev](https://deps.dev). It contains the full resolved dependency graph for NPM, PyPI, Maven, Go, and Cargo, updated weekly. For NPM alone it covers ~3M package versions.

**Key tables we use:**
- `PackageVersionsLatest` — one row per package, with version, license, and linked project info
- `DependentsLatest` — for each package, all packages that depend on it (pre-computed!)
- `DependencyGraphEdgesLatest` — raw directed edges: (package A, version) → depends on → (package B, version)

**Why `DependentsLatest` is valuable:** It pre-computes the transitive dependent count — we don't have to BFS ourselves to know that `lodash` has 47k downstream packages. This is the key metric for prioritising which packages to focus on.

**Cost:** This query scans ~200MB–1GB — well within the free 1TB/month. We always dry-run first.

In [ ]:
import pandas as pd

# Query 1: NPM package metadata from deps.dev
#
# WHY the CTE pattern:
# DependentsLatest has one row per (package, dependent) pair — joining it
# directly to PackageVersionsLatest before aggregating creates a massive
# cartesian product (~51TB scan). Instead we pre-aggregate dependent counts
# in a CTE first (one row per package), then join that small result.

packages_query = """
WITH dependent_counts AS (
  SELECT
    Name,
    COUNT(DISTINCT Dependent.Name) AS dependent_count
  FROM `bigquery-public-data.deps_dev_v1.DependentsLatest`
  WHERE System = 'NPM'
  GROUP BY Name
)
SELECT
  pv.Name                                                                         AS name,
  pv.Version                                                                      AS version,
  pv.Licenses                                                                     AS licenses,
  (SELECT l.URL FROM UNNEST(pv.Links) AS l WHERE l.Label = 'HOMEPAGE'    LIMIT 1) AS homepage,
  (SELECT l.URL FROM UNNEST(pv.Links) AS l WHERE l.Label = 'SOURCE_REPO' LIMIT 1) AS repo_url,
  COALESCE(dc.dependent_count, 0)                                                 AS dependent_count
FROM
  `bigquery-public-data.deps_dev_v1.PackageVersionsLatest` AS pv
LEFT JOIN
  dependent_counts AS dc
  ON pv.Name = dc.Name
WHERE
  pv.System = 'NPM'
ORDER BY
  dependent_count DESC
"""

# Dry run first — check cost before committing
job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry = bq.query(packages_query, job_config=job_config)
gb = dry.total_bytes_processed / 1e9
print(f'Packages query — estimated scan: {gb:.2f} GB  (free under 1TB/month)')
print(f'Estimated cost: ${gb / 1000 * 6.25:.4f}')

In [ ]:
print('Running packages query (1–3 minutes)...')
npm_packages = bq.query(packages_query).to_dataframe()

# Extract GitHub slug from repo URL for joining with GitHub Archive later
npm_packages['github_slug'] = (
    npm_packages['repo_url']
    .str.extract(r'github\.com/([^/]+/[^/]+?)(?:\.git)?/?$', expand=False)
    .str.lower()
)

print(f'NPM packages fetched: {len(npm_packages):,}')
print(f'With GitHub repo:     {npm_packages["github_slug"].notna().sum():,}')
print(f'\nTop 10 by dependent count:')
print(npm_packages.head(10)[['name', 'version', 'dependent_count']].to_string(index=False))

In [ ]:
npm_packages.to_parquet(f'{PROJECT_ROOT}/data/processed/npm_packages.parquet', index=False)
print(f'Saved {len(npm_packages):,} packages → data/processed/npm_packages.parquet')

In [ ]:
# Query 2: NPM dependency edges from deps.dev
# DependencyGraphEdgesLatest has one row per (package→version, dependency→version) edge.
# We deduplicate to package→package level (ignoring specific versions) to keep the graph tractable.
# We only take edges where the SOURCE package is in our top 50k by dependent count.

deps_query = """
WITH top_packages AS (
  -- Limit to the 50k most-depended-upon NPM packages
  SELECT Name
  FROM `bigquery-public-data.deps_dev_v1.DependentsLatest`
  WHERE System = 'NPM'
  GROUP BY Name
  ORDER BY COUNT(DISTINCT Dependent.Name) DESC
  LIMIT 50000
)
SELECT DISTINCT
  e.From.Name   AS src_package,
  e.To.Name     AS dst_package,
  'runtime'     AS dep_kind
FROM
  `bigquery-public-data.deps_dev_v1.DependencyGraphEdgesLatest` AS e
WHERE
  e.System = 'NPM'
  AND e.From.Name IN (SELECT Name FROM top_packages)
  AND e.To.Name   IN (SELECT Name FROM top_packages)
"""

job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry = bq.query(deps_query, job_config=job_config)
gb = dry.total_bytes_processed / 1e9
print(f'Dependencies query — estimated scan: {gb:.2f} GB')

In [ ]:
print('Running dependencies query (2–5 minutes)...')
npm_deps = bq.query(deps_query).to_dataframe()
npm_deps['is_optional'] = False

print(f'Dependency edges fetched: {len(npm_deps):,}')
print(f'Unique source packages:   {npm_deps["src_package"].nunique():,}')
print(f'Unique target packages:   {npm_deps["dst_package"].nunique():,}')
npm_deps.head(3)

In [ ]:
npm_deps.to_parquet(f'{PROJECT_ROOT}/data/processed/npm_dependencies.parquet', index=False)
print(f'Saved {len(npm_deps):,} dependency edges → data/processed/npm_dependencies.parquet')

## 3 — Libraries.io metadata supplement (Kaggle API)

**Why still use Libraries.io?** `deps.dev` gives us the dependency graph but doesn't provide GitHub star counts or a direct `dependent_repos` count (distinct repos that depend on this package, as opposed to packages). Libraries.io has this, and we use it to enrich our package metadata for the report.

**How the Kaggle API works in Colab:**
1. Go to [kaggle.com/settings](https://www.kaggle.com/settings) → API → "Create New Token" — this downloads a `kaggle.json` file.
2. Upload that file using the Colab file uploader (the folder icon in the left sidebar).
3. The cell below moves it to the right location and calls `kaggle datasets download`.

The dataset is ~1GB compressed. It downloads directly into Colab's local disk (not Drive), we filter it, then save only the NPM subset to Drive.

In [ ]:
# Step 1: Upload your kaggle.json via the Colab file picker
from google.colab import files
print('Upload your kaggle.json file (from kaggle.com/settings → API → Create New Token):')
uploaded = files.upload()   # opens the file picker

# Move it to the location the kaggle CLI expects
import shutil, json
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json installed.')

In [ ]:
# Step 2: Download the Libraries.io dataset into Colab's local /tmp (not Drive — it's temporary)
# We only need two CSV files: projects and dependencies
print('Downloading Libraries.io dataset from Kaggle (~1GB, takes 2-4 minutes)...')
!kaggle datasets download librariesdotio/libraries-io -p /tmp/librariesio --unzip
print('Download complete.')
!ls /tmp/librariesio/

In [ ]:
# Step 3: Filter to NPM and extract the columns we need
# The file is large so we use chunked reading and filter each chunk
import glob

# Find the projects file (name may vary by dataset version)
projects_file = glob.glob('/tmp/librariesio/projects*.csv')[0]
print(f'Projects file: {projects_file}')

projects_cols = [
    'Platform', 'Name', 'Repository URL', 'Stars',
    'Dependent Repositories Count', 'Status'
]

# Read in 100k-row chunks and keep only NPM rows
npm_meta_chunks = []
for chunk in pd.read_csv(projects_file, usecols=projects_cols, chunksize=100_000, low_memory=False):
    npm_chunk = chunk[chunk['Platform'] == 'NPM']
    if len(npm_chunk) > 0:
        npm_meta_chunks.append(npm_chunk)

npm_meta = (
    pd.concat(npm_meta_chunks, ignore_index=True)
    .rename(columns={
        'Name': 'name',
        'Repository URL': 'repo_url_lio',
        'Stars': 'stars',
        'Dependent Repositories Count': 'dependent_repos',
        'Status': 'status'
    })
    .drop(columns=['Platform'])
)
npm_meta['name'] = npm_meta['name'].str.lower()

print(f'Libraries.io NPM packages: {len(npm_meta):,}')

In [ ]:
# Merge the Libraries.io metadata into our deps.dev package table
# deps.dev is the authoritative source for dependency data; Libraries.io adds stars/status
npm_packages['name_lower'] = npm_packages['name'].str.lower()
npm_packages = npm_packages.merge(
    npm_meta[['name', 'stars', 'dependent_repos', 'status']],
    left_on='name_lower', right_on='name', how='left',
    suffixes=('', '_lio')
).drop(columns=['name_lio', 'name_lower'])

print(f'Enriched packages: {len(npm_packages):,}')
print(f'With star data:    {npm_packages["stars"].notna().sum():,}')

npm_packages.to_parquet(f'{PROJECT_ROOT}/data/processed/npm_packages.parquet', index=False)
print('Saved enriched npm_packages.parquet')

## 4 — GitHub Archive: contributor commit activity

**Why:** `deps.dev` and Libraries.io tell us about package structure, but not who is actively maintaining a repo. For the Bus Factor metric we need: how many distinct GitHub users committed to each repo in the last 12 months?

The GitHub Archive (`githubarchive.year.*`) records every public GitHub event since 2011. We filter to `PushEvent` (commits pushed to a branch) and group by repo + author.

**Cost awareness:** Each year table is ~200-400GB. Filtering by repo name (the `IN UNNEST` clause) pushes the scan down significantly, but this query will still scan a few GB — still free under the 1TB quota. Check the dry-run estimate before running.

In [ ]:
# Get the GitHub slugs for our top 50k packages (by dependent count)
top_50k = npm_packages.nlargest(50000, 'dependent_count')
npm_github_slugs = (
    top_50k['github_slug']
    .dropna()
    .str.lower()
    .unique()
    .tolist()
)
print(f'Packages with GitHub repos (top 50k): {len(npm_github_slugs):,}')

In [ ]:
# BigQuery supports passing arrays via query parameters — much cleaner than string interpolation
# and avoids any injection risk with unusual package names.

gh_query = """
SELECT
  LOWER(repo.name)   AS github_slug,
  actor.login        AS author_login,
  DATE(created_at)   AS commit_date,
  COUNT(*)           AS push_count
FROM (
  SELECT type, repo, actor, created_at FROM `githubarchive.year.2022`
  UNION ALL
  SELECT type, repo, actor, created_at FROM `githubarchive.year.2023`
  UNION ALL
  SELECT type, repo, actor, created_at FROM `githubarchive.year.2024`
)
WHERE
  type = 'PushEvent'
  AND LOWER(repo.name) IN UNNEST(@slugs)
GROUP BY
  github_slug, author_login, commit_date
"""

job_config = bigquery.QueryJobConfig(
    dry_run=True,
    use_query_cache=False,
    query_parameters=[
        bigquery.ArrayQueryParameter('slugs', 'STRING', npm_github_slugs[:50000])
    ]
)
dry = bq.query(gh_query, job_config=job_config)
gb = dry.total_bytes_processed / 1e9
print(f'GitHub Archive query — estimated scan: {gb:.2f} GB')
print(f'Estimated cost: ${gb / 1000 * 6.25:.4f}  (free under 1TB/month)')

In [ ]:
print('Running GitHub Archive query (2–5 minutes)...')
job_config = bigquery.QueryJobConfig(
    use_query_cache=True,
    query_parameters=[
        bigquery.ArrayQueryParameter('slugs', 'STRING', npm_github_slugs[:50000])
    ]
)
gh_commits = bq.query(gh_query, job_config=job_config).to_dataframe()
gh_commits['commit_date'] = pd.to_datetime(gh_commits['commit_date'])

print(f'Rows returned:    {len(gh_commits):,}')
print(f'Unique repos:     {gh_commits["github_slug"].nunique():,}')
print(f'Unique authors:   {gh_commits["author_login"].nunique():,}')
print(f'Date range:       {gh_commits["commit_date"].min().date()} → {gh_commits["commit_date"].max().date()}')
gh_commits.head(3)

In [ ]:
gh_commits.to_parquet(f'{PROJECT_ROOT}/data/processed/gh_commits.parquet', index=False)
print(f'Saved {len(gh_commits):,} rows → data/processed/gh_commits.parquet')

## 5 — OSV: NPM vulnerability feed

**Why OSV over NVD?** The National Vulnerability Database (NVD) is the traditional CVE source, but it's slow to update (often months behind) and its package-name mapping is inconsistent. OSV (Open Source Vulnerabilities) is maintained by Google specifically for open-source ecosystems. The npm feed is updated daily and uses npm package names directly — no fuzzy matching needed.

The feed is a zip of individual JSON files (one per CVE), publicly accessible from a GCS bucket with no authentication.

**Note:** `deps.dev`'s `AdvisoriesLatest` table also exposes OSV data via BigQuery. We download the raw JSON here because we want the full `affected_ranges` for version matching in notebook 04.

In [ ]:
import urllib.request
import zipfile
import json
from pathlib import Path

OSV_URL = 'https://osv-vulnerabilities.storage.googleapis.com/npm/all.zip'
OSV_ZIP = f'{PROJECT_ROOT}/data/raw/osv_npm.zip'
OSV_DIR = f'{PROJECT_ROOT}/data/raw/osv_npm'

print('Downloading OSV npm feed (~30MB)...')
urllib.request.urlretrieve(OSV_URL, OSV_ZIP)

print('Extracting...')
with zipfile.ZipFile(OSV_ZIP, 'r') as z:
    z.extractall(OSV_DIR)

osv_files = list(Path(OSV_DIR).glob('*.json'))
print(f'Extracted {len(osv_files):,} vulnerability JSON files')

In [ ]:
def parse_osv_file(path):
    with open(path) as f:
        data = json.load(f)

    vuln_id  = data.get('id', '')
    summary  = data.get('summary', '')
    severity = data.get('database_specific', {}).get('severity', 'UNKNOWN')
    published = data.get('published', '')

    rows = []
    for affected in data.get('affected', []):
        pkg = affected.get('package', {})
        rows.append({
            'vuln_id':        vuln_id,
            'package_name':   pkg.get('name', '').lower(),
            'ecosystem':      pkg.get('ecosystem', ''),
            'severity':       severity,
            'summary':        summary,
            'published':      published,
            'affected_ranges': json.dumps(affected.get('ranges', []))
        })
    return rows

print('Parsing OSV files...')
all_rows = []
for path in osv_files:
    try:
        all_rows.extend(parse_osv_file(path))
    except Exception as e:
        print(f'  Warning: {path.name}: {e}')

osv_df = pd.DataFrame(all_rows)
osv_df['published'] = pd.to_datetime(osv_df['published'], errors='coerce')

print(f'Total OSV records:              {len(osv_df):,}')
print(f'Unique packages with CVEs:      {osv_df["package_name"].nunique():,}')
print(f'Severity breakdown:')
print(osv_df['severity'].value_counts())

In [ ]:
osv_df.to_parquet(f'{PROJECT_ROOT}/data/processed/osv_npm.parquet', index=False)
print(f'Saved {len(osv_df):,} OSV records → data/processed/osv_npm.parquet')

## 6 — npm registry: maintainer data (Orphan Risk)

**Why the npm registry directly?** Neither `deps.dev` nor Libraries.io track npm's `maintainers` field — the list of npm accounts with publish rights. This is the most critical chokepoint for supply chain attacks: the `event-stream` hijack succeeded because the original maintainer handed npm publish rights to a stranger.

We query the npm registry REST API for the top 5k packages by `dependent_count`. We use `asyncio` + `aiohttp` to parallelise the requests (20 at a time) so 5k packages takes ~5 minutes instead of ~1.5 hours sequential.

In [ ]:
import asyncio
import aiohttp
import nest_asyncio
from tqdm.auto import tqdm

nest_asyncio.apply()   # needed because Colab already runs an event loop

top_packages = (
    npm_packages
    .nlargest(5000, 'dependent_count')
    ['name']
    .tolist()
)
print(f'Fetching npm registry data for top {len(top_packages):,} packages by dependent count')
print('Sample:', top_packages[:5])

In [ ]:
NPM_REGISTRY = 'https://registry.npmjs.org'
CONCURRENCY  = 20
DELAY_MS     = 50

async def fetch_package(session, name, semaphore):
    url = f'{NPM_REGISTRY}/{name}'
    # The abbreviated manifest Accept header returns a smaller payload (no README, no dist)
    headers = {'Accept': 'application/vnd.npm.install-v1+json'}
    async with semaphore:
        try:
            async with session.get(url, headers=headers, timeout=aiohttp.ClientTimeout(total=10)) as resp:
                if resp.status != 200:
                    return None
                data = await resp.json(content_type=None)
                return {
                    'name':             name,
                    'npm_maintainers':  [m.get('name', '') for m in data.get('maintainers', [])],
                    'maintainer_count': len(data.get('maintainers', [])),
                    'npm_last_modified': data.get('time', {}).get('modified', ''),
                    'latest_version':   data.get('dist-tags', {}).get('latest', '')
                }
        except Exception:
            return None

async def fetch_all(names):
    semaphore = asyncio.Semaphore(CONCURRENCY)
    results = []
    async with aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit=CONCURRENCY)) as session:
        tasks = [fetch_package(session, name, semaphore) for name in names]
        for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc='npm registry'):
            result = await coro
            if result:
                results.append(result)
            await asyncio.sleep(DELAY_MS / 1000)
    return results

print(f'Fetching (concurrency={CONCURRENCY}, {DELAY_MS}ms delay)...')
maintainer_records = asyncio.run(fetch_all(top_packages))
print(f'Fetched: {len(maintainer_records):,} / {len(top_packages):,}')

In [ ]:
npm_maintainers = pd.DataFrame(maintainer_records)
npm_maintainers['npm_last_modified'] = pd.to_datetime(npm_maintainers['npm_last_modified'], errors='coerce')

print(f'Packages fetched: {len(npm_maintainers):,}')
print(f'Solo maintainers (count=1): {(npm_maintainers["maintainer_count"] == 1).sum():,}')
print(f'≤2 maintainers:             {(npm_maintainers["maintainer_count"] <= 2).sum():,}')

# Save (convert list column to JSON string for parquet)
npm_maintainers_save = npm_maintainers.copy()
npm_maintainers_save['npm_maintainers'] = npm_maintainers_save['npm_maintainers'].apply(json.dumps)
npm_maintainers_save.to_parquet(f'{PROJECT_ROOT}/data/processed/npm_maintainers.parquet', index=False)
print(f'Saved → data/processed/npm_maintainers.parquet')

## 7 — Build the 10k development sample

**Why a sample?** Running PySpark GraphFrames PageRank on 50k nodes takes 10–20 minutes in Colab. On 10k nodes it takes 1–2 minutes. During development (notebooks 02–04) we iterate on the 10k sample so we can test ideas quickly. The final full run happens in notebook 06 when the code is proven.

We pick the top 10k packages by `dependent_count` — the packages most other packages depend on. This is the most structurally important slice of the graph.

In [ ]:
sample_packages = (
    npm_packages
    .nlargest(10_000, 'dependent_count')
    .reset_index(drop=True)
)
sample_names = set(sample_packages['name'].str.lower())

# Filter dependency edges: keep only edges where BOTH ends are in our sample
sample_deps = npm_deps[
    npm_deps['src_package'].str.lower().isin(sample_names) &
    npm_deps['dst_package'].str.lower().isin(sample_names)
].reset_index(drop=True)

# Filter commits: keep only repos in our sample
sample_slugs = set(sample_packages['github_slug'].dropna().str.lower())
sample_commits = gh_commits[
    gh_commits['github_slug'].isin(sample_slugs)
].reset_index(drop=True)

print(f'Sample packages:    {len(sample_packages):,}')
print(f'Sample dep edges:   {len(sample_deps):,}')
print(f'Sample commit rows: {len(sample_commits):,}')

In [ ]:
sample_packages.to_parquet(f'{PROJECT_ROOT}/data/sample/npm_packages_10k.parquet', index=False)
sample_deps.to_parquet(f'{PROJECT_ROOT}/data/sample/npm_dependencies_10k.parquet', index=False)
sample_commits.to_parquet(f'{PROJECT_ROOT}/data/sample/gh_commits_10k.parquet', index=False)
print('Sample files saved to data/sample/')

## 8 — Sanity checks

Run these before moving to notebook 02. If any check fails, something went wrong in ingestion.

In [ ]:
# Check 1: Well-known packages should be in our dataset
known = ['react', 'lodash', 'express', 'typescript', 'axios', 'webpack']
found   = [p for p in known if p in sample_names]
missing = [p for p in known if p not in sample_names]

print('CHECK 1 — known packages in 10k sample:')
print(f'  Found:   {found}')
if missing:
    print(f'  MISSING: {missing}  ← investigate')
else:
    print('  All present ✓')

In [ ]:
# Check 2: Top packages by dependent count should be recognisable
print('CHECK 2 — top 10 by dependent_count (should be well-known packages):')
print(
    sample_packages.head(10)[['name', 'dependent_count', 'stars']]
    .to_string(index=False)
)

In [ ]:
# Check 3: lodash should have CVEs
lodash_cves = osv_df[osv_df['package_name'] == 'lodash']
print(f'CHECK 3 — lodash CVEs: {len(lodash_cves)} records')
if len(lodash_cves) > 0:
    print(lodash_cves[['vuln_id', 'severity', 'summary']].to_string())
    print('  ✓')
else:
    print('  WARNING: no CVEs for lodash — check OSV parsing')

In [ ]:
# Check 4: npm maintainer spot-check
print('CHECK 4 — npm maintainer spot-check:')
spot = npm_maintainers[npm_maintainers['name'].isin(['react', 'lodash', 'express'])]
print(spot[['name', 'maintainer_count', 'npm_last_modified']].to_string(index=False))

print('\nTop 5 solo-maintained packages by dependent count:')
solo = (
    npm_maintainers[npm_maintainers['maintainer_count'] == 1]
    .merge(npm_packages[['name', 'dependent_count']], on='name')
    .nlargest(5, 'dependent_count')
    [['name', 'dependent_count', 'maintainer_count', 'npm_last_modified']]
)
print(solo.to_string(index=False))

In [ ]:
# Final summary
print('=' * 50)
print('INGESTION COMPLETE')
print('=' * 50)
for root, dirs, files in os.walk(f'{PROJECT_ROOT}/data'):
    level  = root.replace(f'{PROJECT_ROOT}/data', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for fname in sorted(files):
        fpath   = os.path.join(root, fname)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'{indent}  {fname}  ({size_mb:.1f} MB)')
print('\nNext: open 02_graph_construction.ipynb')